# `xr.DataTree` reference

Worked examples of every tree-only attribute on `xarray.DataTree` (the ones `xr.Dataset` doesn't have),
plus the shared-name methods that behave differently on a tree.

Built against xarray 2026.7. Uses a small synthetic LESFMIP-style tree, laid out as `/<model>/<experiment>`,
so every cell runs without JASMIN.

## 0. Setup: a toy LESFMIP tree

In [1]:
import numpy as np
import pandas as pd
import xarray as xr

xr.set_options(display_expand_data=False)
print(xr.__version__)

2026.7.0


The tree is deliberately a bit messy, like the real thing:

- `year` is a coordinate on the **root** and gets *inherited* by every node below.
- Model nodes carry attrs but no data (a "hollow" tree: data only in the leaves).
- It's **ragged**: MIROC6 has no `hist-sol`, and `/MIROC6/hist-nat` exists but is empty.

In [2]:
rng = np.random.default_rng(0)
years = np.arange(1850, 2021)
n_mem = 3

# rough forced signals (K), just for something plausible to look at
t = (years - 1850) / 170
signals = {
    "historical": 1.1 * t**2.5,
    "hist-GHG":   1.5 * t**2.5,
    "hist-aer":  -0.5 * t**2,
    "hist-sol":   0.05 * np.sin(2 * np.pi * (years - 1850) / 11),
}

def make_run(signal, noise=0.15):
    tas = signal + rng.normal(0, noise, size=(n_mem, years.size))
    return xr.Dataset(
        {"tas": (("member", "year"), tas)},
        coords={"member": np.arange(1, n_mem + 1)},
    )

experiments = {
    "ACCESS-ESM1-5": ["historical", "hist-GHG", "hist-aer", "hist-sol"],
    "CanESM5":       ["historical", "hist-GHG", "hist-aer", "hist-sol"],
    "MIROC6":        ["historical", "hist-GHG", "hist-aer"],   # ragged
}
institutions = {"ACCESS-ESM1-5": "CSIRO", "CanESM5": "CCCma", "MIROC6": "MIROC"}

d = {"/": xr.Dataset(coords={"year": years}, attrs={"project": "LESFMIP (toy)"})}
for model, exps in experiments.items():
    d[f"/{model}"] = xr.Dataset(attrs={"institution": institutions[model]})
    for exp in exps:
        d[f"/{model}/{exp}"] = make_run(signals[exp])
d["/MIROC6/hist-nat"] = None   # an empty node, for prune/has_data demos

dt = xr.DataTree.from_dict(d)
dt

<xarray.DataTree>
Group: /
│   Dimensions:  (year: 171)
│   Coordinates:
│     * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2017 2018 2019 2020
│   Attributes:
│       project:  LESFMIP (toy)
├── Group: /ACCESS-ESM1-5
│   │   Attributes:
│   │       institution:  CSIRO
│   ├── Group: /ACCESS-ESM1-5/historical
│   │       Dimensions:  (member: 3, year: 171)
│   │       Coordinates:
│   │         * member   (member) int64 24B 1 2 3
│   │       Data variables:
│   │           tas      (member, year) float64 4kB 0.01886 -0.01981 0.09608 ... 1.105 1.018
│   ├── Group: /ACCESS-ESM1-5/hist-GHG
│   │       Dimensions:  (member: 3, year: 171)
│   │       Coordinates:
│   │         * member   (member) int64 24B 1 2 3
│   │       Data variables:
│   │           tas      (member, year) float64 4kB -0.02002 0.1947 -0.1452 ... 1.551 1.742
│   ├── Group: /ACCESS-ESM1-5/hist-aer
│   │       Dimensions:  (member: 3, year: 171)
│   │       Coordinates:
│   │         * member   (member) int64 24B 1 2 3
│   │       Data variables:
│   │           tas      (member, year) float64 4kB -0.1173 -0.01424 ... -0.2865 -0.5162
│   └── Group: /ACCESS-ESM1-5/hist-sol
│           Dimensions:  (member: 3, year: 171)
│           Coordinates:
│             * member   (member) int64 24B 1 2 3
│           Data variables:
│               tas      (member, year) float64 4kB -0.08564 0.06196 ... 0.08559 0.2133
├── Group: /CanESM5
│   │   Attributes:
│   │       institution:  CCCma
│   ├── Group: /CanESM5/historical
│   │       Dimensions:  (member: 3, year: 171)
│   │       Coordinates:
│   │         * member   (member) int64 24B 1 2 3
│   │       Data variables:
│   │           tas      (member, year) float64 4kB -0.1353 0.01052 0.07176 ... 1.457 1.292
│   ├── Group: /CanESM5/hist-GHG
│   │       Dimensions:  (member: 3, year: 171)
│   │       Coordinates:
│   │         * member   (member) int64 24B 1 2 3
│   │       Data variables:
│   │           tas      (member, year) float64 4kB -0.2017 -0.1928 0.0402 ... 1.58 1.743
│   ├── Group: /CanESM5/hist-aer
│   │       Dimensions:  (member: 3, year: 171)
│   │       Coordinates:
│   │         * member   (member) int64 24B 1 2 3
│   │       Data variables:
│   │           tas      (member, year) float64 4kB -0.05611 0.1684 ... -0.5575 -0.4366
│   └── Group: /CanESM5/hist-sol
│           Dimensions:  (member: 3, year: 171)
│           Coordinates:
│             * member   (member) int64 24B 1 2 3
│           Data variables:
│               tas      (member, year) float64 4kB 0.09112 0.09669 0.236 ... 0.0342 0.1852
└── Group: /MIROC6
    │   Attributes:
    │       institution:  MIROC
    ├── Group: /MIROC6/historical
    │       Dimensions:  (member: 3, year: 171)
    │       Coordinates:
    │         * member   (member) int64 24B 1 2 3
    │       Data variables:
    │           tas      (member, year) float64 4kB 0.001865 -0.06814 ... 1.218 1.187
    ├── Group: /MIROC6/hist-GHG
    │       Dimensions:  (member: 3, year: 171)
    │       Coordinates:
    │         * member   (member) int64 24B 1 2 3
    │       Data variables:
    │           tas      (member, year) float64 4kB 0.146 0.027 0.2367 ... 1.474 1.394 1.381
    ├── Group: /MIROC6/hist-aer
    │       Dimensions:  (member: 3, year: 171)
    │       Coordinates:
    │         * member   (member) int64 24B 1 2 3
    │       Data variables:
    │           tas      (member, year) float64 4kB 0.1372 0.1449 0.3074 ... -0.6822 -0.5315
    └── Group: /MIROC6/hist-nat

In [3]:
# handy references used below
leaf = dt["ACCESS-ESM1-5/hist-GHG"]      # paths work in __getitem__
model_node = dt["ACCESS-ESM1-5"]
empty_node = dt["MIROC6/hist-nat"]

## 1. Getting at the data in a node

| attribute | what it gives you |
|---|---|
| `.dataset` / `.ds` | read-only `DatasetView` of this node, **including inherited coords** |
| `.to_dataset(inherit=...)` | a real, mutable `Dataset` copy |
| `.has_data` | node has any variables **or coords defined on it** |
| `.has_attrs` | node has any attrs |
| `.is_empty` | no variables *and* no attrs (ignores children) |
| `.name` | this node's key under its parent |

### `.dataset` / `.ds`: read-only view (they're aliases)

In [4]:
leaf.ds   # note `year` shows up even though it was only defined on the root

<xarray.DatasetView> Size: 5kB
Dimensions:  (member: 3, year: 171)
Coordinates:
  * member   (member) int64 24B 1 2 3
  * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2017 2018 2019 2020
Data variables:
    tas      (member, year) float64 4kB -0.02002 0.1947 -0.1452 ... 1.551 1.742

In [5]:
# it's immutable: assigning raises
try:
    leaf.ds["tas2"] = leaf.ds.tas * 2
except Exception as e:
    print(type(e).__name__, "-", e)

AttributeError - Mutation of the DatasetView is not allowed, please use `.__setitem__` on the wrapping DataTree node, or use `dt.to_dataset()` if you want a mutable dataset. If calling this from within `map_over_datasets`,use `.copy()` first to get a mutable version of the input dataset.


### `.to_dataset(inherit=...)`: mutable copy
- `True` / `"indexes"` (default): inherit indexed coords from parents
- `"all_coords"`: also inherit non-index coords
- `False`: only what's defined on this node

In [6]:
ds = leaf.to_dataset()
ds["tas2"] = ds.tas * 2          # fine, it's a normal Dataset
print("inherit=True :", list(leaf.to_dataset().coords))
print("inherit=False:", list(leaf.to_dataset(inherit=False).coords))   # year dim remains, coord gone

inherit=True : ['year', 'member']
inherit=False: ['member']


### `.has_data`, `.has_attrs`, `.is_empty`, `.name`

In [7]:
for node in [dt, model_node, leaf, empty_node]:
    print(f"{node.path:28s} name={node.name!s:14s} has_data={node.has_data!s:5s} "
          f"has_attrs={node.has_attrs!s:5s} is_empty={node.is_empty}")

/                            name=None           has_data=True  has_attrs=True  is_empty=False
/ACCESS-ESM1-5               name=ACCESS-ESM1-5  has_data=False has_attrs=True  is_empty=False
/ACCESS-ESM1-5/hist-GHG      name=hist-GHG       has_data=True  has_attrs=False is_empty=False
/MIROC6/hist-nat             name=hist-nat       has_data=False has_attrs=False is_empty=True


- Root's `name` is `None`.
- **Watch out:** `has_data` counts *coordinates defined on that node*, not just data variables. The root only holds
  the `year` coordinate, yet `has_data` is True. Inherited coords don't count, so the model nodes are False.
- Model nodes have attrs, so they're not `is_empty` even though they hold no data.
- If you mean "has actual data variables", test `bool(node.data_vars)` or `"tas" in node.data_vars`.

In [8]:
for node in [dt, model_node, leaf]:
    print(f"{node.path:24s} has_data={node.has_data!s:5s} data_vars={list(node.data_vars)}")

/                        has_data=True  data_vars=[]
/ACCESS-ESM1-5           has_data=False data_vars=[]
/ACCESS-ESM1-5/hist-GHG  has_data=True  data_vars=['tas']


## 2. Where a node sits in the tree

| attribute | what it gives you |
|---|---|
| `.parent` | node directly above (`None` at root) |
| `.root` | top of the tree |
| `.children` | dict `{name: DataTree}` of immediate children |
| `.siblings` | dict of the parent's *other* children |
| `.parents` | tuple of all nodes above, closest first |
| `.path` | absolute path string |
| `.relative_to(other)` | path of this node relative to another node |
| `.is_root`, `.is_leaf` | booleans |
| `.level` | levels below root (root = 0) |
| `.depth` | max level in the whole tree |
| `.width` | number of nodes at this node's level, across the whole tree |

In [9]:
print("parent  :", leaf.parent.path)
print("root    :", leaf.root.path)
print("children:", list(model_node.children))
print("siblings:", list(leaf.siblings))
print("parents :", [p.path for p in leaf.parents])
print("path    :", leaf.path)

parent  : /ACCESS-ESM1-5
root    : /
children: ['historical', 'hist-GHG', 'hist-aer', 'hist-sol']
siblings: ['historical', 'hist-aer', 'hist-sol']
parents : ['/ACCESS-ESM1-5', '/']
path    : /ACCESS-ESM1-5/hist-GHG


### `.relative_to(other)`: *this* node's path, measured from `other`

In [10]:
print(leaf.relative_to(dt))            # 'ACCESS-ESM1-5/hist-GHG'  <- what tree_to_dataset uses
print(leaf.relative_to(model_node))    # 'hist-GHG'
print(leaf.relative_to(dt["CanESM5/historical"]))   # '../../ACCESS-ESM1-5/hist-GHG'

ACCESS-ESM1-5/hist-GHG
hist-GHG
../../ACCESS-ESM1-5/hist-GHG


### Shape of the tree

In [11]:
for node in [dt, model_node, leaf]:
    print(f"{node.path:24s} is_root={node.is_root!s:5s} is_leaf={node.is_leaf!s:5s} "
          f"level={node.level} depth={node.depth} width={node.width}")

/                        is_root=True  is_leaf=False level=0 depth=2 width=1
/ACCESS-ESM1-5           is_root=False is_leaf=False level=1 depth=2 width=3
/ACCESS-ESM1-5/hist-GHG  is_root=False is_leaf=True  level=2 depth=2 width=12


- `depth` is a property of the *tree*, so it's 2 wherever you ask.
- `width` at the leaf level is 12: every experiment node across all models (including the empty `hist-nat`).

## 3. Iterating over the tree

| attribute | yields | order |
|---|---|---|
| `.subtree` | self + everything below | breadth-first |
| `.subtree_with_keys` | `(relative_path, node)` pairs, self is `"."` | breadth-first |
| `.descendants` | everything below, **excluding self** | depth-first |
| `.leaves` | nodes with no children | |
| `.groups` | tuple of every path string | |

In [12]:
print("subtree    :", [n.path for n in model_node.subtree])
print("descendants:", [n.path for n in model_node.descendants])

subtree    : ['/ACCESS-ESM1-5', '/ACCESS-ESM1-5/historical', '/ACCESS-ESM1-5/hist-GHG', '/ACCESS-ESM1-5/hist-aer', '/ACCESS-ESM1-5/hist-sol']
descendants: ['/ACCESS-ESM1-5/historical', '/ACCESS-ESM1-5/hist-GHG', '/ACCESS-ESM1-5/hist-aer', '/ACCESS-ESM1-5/hist-sol']


In [13]:
# subtree_with_keys is the most useful one for your own loops
for key, node in dt.subtree_with_keys:
    if "tas" in node.data_vars:     # not has_data: the root has a coord, so has_data is True there
        print(f"{key:26s} tas mean = {float(node.ds.tas.mean()):.3f}")

ACCESS-ESM1-5/historical   tas mean = 0.312
ACCESS-ESM1-5/hist-GHG     tas mean = 0.420
ACCESS-ESM1-5/hist-aer     tas mean = -0.160
ACCESS-ESM1-5/hist-sol     tas mean = -0.008
CanESM5/historical         tas mean = 0.308
CanESM5/hist-GHG           tas mean = 0.427
CanESM5/hist-aer           tas mean = -0.163
CanESM5/hist-sol           tas mean = 0.006
MIROC6/historical          tas mean = 0.324
MIROC6/hist-GHG            tas mean = 0.437
MIROC6/hist-aer            tas mean = -0.172


In [14]:
print("leaves:", [n.path for n in dt.leaves])
print()
print("groups:", dt.groups)

leaves: ['/ACCESS-ESM1-5/historical', '/ACCESS-ESM1-5/hist-GHG', '/ACCESS-ESM1-5/hist-aer', '/ACCESS-ESM1-5/hist-sol', '/CanESM5/historical', '/CanESM5/hist-GHG', '/CanESM5/hist-aer', '/CanESM5/hist-sol', '/MIROC6/historical', '/MIROC6/hist-GHG', '/MIROC6/hist-aer', '/MIROC6/hist-nat']

groups: ('/', '/ACCESS-ESM1-5', '/CanESM5', '/MIROC6', '/ACCESS-ESM1-5/historical', '/ACCESS-ESM1-5/hist-GHG', '/ACCESS-ESM1-5/hist-aer', '/ACCESS-ESM1-5/hist-sol', '/CanESM5/historical', '/CanESM5/hist-GHG', '/CanESM5/hist-aer', '/CanESM5/hist-sol', '/MIROC6/historical', '/MIROC6/hist-GHG', '/MIROC6/hist-aer', '/MIROC6/hist-nat')


Note `/MIROC6/hist-nat` is a leaf even though it's empty. That's why `tree_to_dataset` filters out empty leaves.

## 4. Selecting parts of the tree

All return a **new tree**. Intermediate nodes are kept where needed to reach surviving leaves.

| method | keeps |
|---|---|
| `.match(pattern)` | nodes whose path matches a unix glob |
| `.filter(func)` | nodes where `func(node)` is True |
| `.filter_like(other)` | nodes whose paths also exist in `other` |
| `.prune()` | nodes with data (drops empty ones) |
| `.drop_nodes(names)` | everything except the named *immediate* children |

### `.match(pattern)`

In [15]:
dt.match("*/hist-GHG").groups

('/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/MIROC6',
 '/ACCESS-ESM1-5/hist-GHG',
 '/CanESM5/hist-GHG',
 '/MIROC6/hist-GHG')

In [16]:
dt.match("*/hist-*").groups       # all single-forcing runs

('/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/MIROC6',
 '/ACCESS-ESM1-5/hist-GHG',
 '/ACCESS-ESM1-5/hist-aer',
 '/ACCESS-ESM1-5/hist-sol',
 '/CanESM5/hist-GHG',
 '/CanESM5/hist-aer',
 '/CanESM5/hist-sol',
 '/MIROC6/hist-GHG',
 '/MIROC6/hist-aer',
 '/MIROC6/hist-nat')

In [17]:
dt.match("CanESM5/*").groups

('/',
 '/CanESM5',
 '/CanESM5/historical',
 '/CanESM5/hist-GHG',
 '/CanESM5/hist-aer',
 '/CanESM5/hist-sol')

### `.filter(func)`: arbitrary conditions on the node

In [18]:
# only nodes that actually hold tas
dt.filter(lambda n: "tas" in n.data_vars).groups

('/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/MIROC6',
 '/ACCESS-ESM1-5/historical',
 '/ACCESS-ESM1-5/hist-GHG',
 '/ACCESS-ESM1-5/hist-aer',
 '/ACCESS-ESM1-5/hist-sol',
 '/CanESM5/historical',
 '/CanESM5/hist-GHG',
 '/CanESM5/hist-aer',
 '/CanESM5/hist-sol',
 '/MIROC6/historical',
 '/MIROC6/hist-GHG',
 '/MIROC6/hist-aer')

In [19]:
# anything you can compute from the node works, e.g. warming > 1 K at the end
warm = dt.filter(lambda n: "tas" in n.data_vars and float(n.ds.tas.isel(year=-1).mean()) > 1.0)
warm.groups

('/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/MIROC6',
 '/ACCESS-ESM1-5/historical',
 '/ACCESS-ESM1-5/hist-GHG',
 '/CanESM5/historical',
 '/CanESM5/hist-GHG',
 '/MIROC6/historical',
 '/MIROC6/hist-GHG')

### `.filter_like(other)`: cut down to a shared set of paths
e.g. keep only the model/experiment combos that exist in some other tree (obs-constrained subset, another variable, etc.)

In [20]:
other = xr.DataTree.from_dict({
    "/ACCESS-ESM1-5/historical": None,
    "/MIROC6/historical": None,
    "/MIROC6/hist-aer": None,
})
dt.filter_like(other).groups

('/',
 '/ACCESS-ESM1-5',
 '/MIROC6',
 '/ACCESS-ESM1-5/historical',
 '/MIROC6/historical',
 '/MIROC6/hist-aer')

### `.prune()`: drop empty nodes

In [21]:
print("before:", "/MIROC6/hist-nat" in dt.groups)
print("after :", "/MIROC6/hist-nat" in dt.prune().groups)

before: True
after : False


### `.drop_nodes(names)`: only works on immediate children

In [22]:
dt.drop_nodes("MIROC6").groups

('/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/ACCESS-ESM1-5/historical',
 '/ACCESS-ESM1-5/hist-GHG',
 '/ACCESS-ESM1-5/hist-aer',
 '/ACCESS-ESM1-5/hist-sol',
 '/CanESM5/historical',
 '/CanESM5/hist-GHG',
 '/CanESM5/hist-aer',
 '/CanESM5/hist-sol')

In [23]:
# to drop deeper nodes, call it on the parent...
try:
    dt.drop_nodes("MIROC6/hist-nat")
except KeyError as e:
    print("KeyError:", e)

# ...or use match/filter/prune instead
dt.filter(lambda n: n.name != "hist-sol").groups

KeyError: "Cannot drop all nodes - nodes {'MIROC6/hist-nat'} not present"


('/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/MIROC6',
 '/ACCESS-ESM1-5/historical',
 '/ACCESS-ESM1-5/hist-GHG',
 '/ACCESS-ESM1-5/hist-aer',
 '/CanESM5/historical',
 '/CanESM5/hist-GHG',
 '/CanESM5/hist-aer',
 '/MIROC6/historical',
 '/MIROC6/hist-GHG',
 '/MIROC6/hist-aer',
 '/MIROC6/hist-nat')

## 5. Comparing and relating trees

| method | question it answers |
|---|---|
| `.isomorphic(other)` | same set of paths? (data not checked). Needed for `tree1 - tree2` |
| `.same_tree(other)` | are these two nodes in the same tree object? |
| `.find_common_ancestor(other)` | closest node above both |

In [24]:
clim = dt.sel(year=slice(1850, 1900)).mean("year")   # same paths, different data

print("dt ~ clim              :", dt.isomorphic(clim))
print("dt ~ dt.match(...)     :", dt.isomorphic(dt.match("*/historical")))
print("same values ignored    :", dt.isomorphic(dt * 0))

dt ~ clim              : True
dt ~ dt.match(...)     : False
same values ignored    : True


In [25]:
a = dt["ACCESS-ESM1-5/hist-GHG"]
b = dt["ACCESS-ESM1-5/hist-aer"]
c = dt["MIROC6/historical"]

print("same_tree(a, b)          :", a.same_tree(b))
print("same_tree(a, clim.*)     :", a.same_tree(clim["ACCESS-ESM1-5/hist-GHG"]))
print("common ancestor a, b     :", a.find_common_ancestor(b).path)
print("common ancestor a, c     :", a.find_common_ancestor(c).path)

same_tree(a, b)          : True
same_tree(a, clim.*)     : False
common ancestor a, b     : /ACCESS-ESM1-5
common ancestor a, c     : /


## 6. Applying functions and editing structure

### `.map_over_datasets(func, *args, kwargs=None)`
Calls `func(ds)` on every node's dataset and rebuilds a same-shaped tree from the results.
`func` must return a `Dataset` (or `None`), not a `DataArray`.

**Gotcha:** it's called on *every* node, including the empty root and model nodes. This fails:

In [26]:
try:
    dt.map_over_datasets(lambda ds: ds.tas.mean().to_dataset(name="tas"))
except AttributeError as e:
    print("AttributeError:", str(e).splitlines()[0])

AttributeError: 'DatasetView' object has no attribute 'tas'


So guard for empty nodes (this is what your `@skip_empty` decorator was doing):

In [27]:
def ensemble_mean(ds):
    if "tas" not in ds:
        return ds
    return ds.mean("member")

dt.map_over_datasets(ensemble_mean)["CanESM5/hist-GHG"]

<xarray.DataTree 'hist-GHG'>
Group: /CanESM5/hist-GHG
    Dimensions:  (year: 171)
    Inherited coordinates:
      * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2017 2018 2019 2020
    Data variables:
        tas      (year) float64 1kB -0.07993 -0.01331 0.0465 ... 1.479 1.419 1.567

In [28]:
# extra positional args go through *args, keyword args through kwargs= (a dict, not **)
def rolling_mean(ds, window, center=False):
    if "tas" not in ds:
        return ds
    return ds.rolling(year=window, center=center).mean()

dt.map_over_datasets(rolling_mean, 11, kwargs={"center": True})["CanESM5/hist-GHG"]

<xarray.DataTree 'hist-GHG'>
Group: /CanESM5/hist-GHG
    Dimensions:  (year: 171, member: 3)
    Coordinates:
      * member   (member) int64 24B 1 2 3
    Inherited coordinates:
      * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2017 2018 2019 2020
    Data variables:
        tas      (member, year) float64 4kB nan nan nan nan nan ... nan nan nan nan

### `xr.map_over_datasets(func, tree1, tree2, ...)`: node-by-node over several trees
The trees must be isomorphic. `func` receives one dataset from each tree at the same path.

First build a 1850–1900 baseline tree. It's done with `map_over_datasets` rather than `dt.sel(...).mean("year")`
for a reason explained in section 7 (inherited coords).

In [29]:
def climatology(ds, start, end):
    if "tas" not in ds:
        return ds
    return ds.sel(year=slice(start, end)).mean("year")

baseline = dt.map_over_datasets(climatology, 1850, 1900)
baseline["CanESM5/historical"].ds

<xarray.DatasetView> Size: 1kB
Dimensions:  (member: 3, year: 171)
Coordinates:
  * member   (member) int64 24B 1 2 3
  * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2017 2018 2019 2020
Data variables:
    tas      (member) float64 24B 0.009076 0.01507 0.008176

In [30]:
def anomaly(ds, base):
    if "tas" not in ds:
        return ds
    return ds - base

anoms = xr.map_over_datasets(anomaly, dt, baseline)
float(anoms["ACCESS-ESM1-5/historical"].ds.tas.sel(year=slice(1850, 1900)).mean())   # ~0

-7.256359638072919e-19

For simple arithmetic you don't even need it; operators work node-wise on isomorphic trees:

In [31]:
anoms2 = dt - baseline
anoms2.identical(anoms)

True

### `.orphan()`: detach a node from its parent, **in place**
It mutates the tree, so work on a copy.

In [32]:
tmp = dt.copy()
node = tmp["MIROC6/hist-nat"]
node.orphan()

print("still in tmp?  ", "/MIROC6/hist-nat" in tmp.groups)
print("node is root?  ", node.is_root, "| path:", node.path)
print("original dt ok?", "/MIROC6/hist-nat" in dt.groups)

still in tmp?   False
node is root?   True | path: /
original dt ok? True


Related: you add/replace nodes with ordinary item assignment, which creates intermediate nodes as needed.

In [33]:
tmp["UKESM1-0-LL/historical"] = make_run(signals["historical"])
[g for g in tmp.groups if "UKESM" in g]

['/UKESM1-0-LL', '/UKESM1-0-LL/historical']

## 7. Shared names that behave differently on a tree

These don't show up in a `dir()` diff because `Dataset` has a method with the same name.

### Dataset methods are applied to every node

In [34]:
# one line, whole tree: subset, reduce, and it keeps the structure
recent = dt.sel(year=slice(1990, None)).mean(["year", "member"])
recent

<xarray.DataTree>
Group: /
│   Attributes:
│       project:  LESFMIP (toy)
├── Group: /ACCESS-ESM1-5
│   │   Dimensions:  (year: 31)
│   │   Coordinates:
│   │     * year     (year) int64 248B 1990 1991 1992 1993 1994 ... 2017 2018 2019 2020
│   │   Attributes:
│   │       institution:  CSIRO
│   ├── Group: /ACCESS-ESM1-5/historical
│   │       Dimensions:  ()
│   │       Data variables:
│   │           tas      float64 8B 0.8614
│   ├── Group: /ACCESS-ESM1-5/hist-GHG
│   │       Dimensions:  ()
│   │       Data variables:
│   │           tas      float64 8B 1.209
│   ├── Group: /ACCESS-ESM1-5/hist-aer
│   │       Dimensions:  ()
│   │       Data variables:
│   │           tas      float64 8B -0.4001
│   └── Group: /ACCESS-ESM1-5/hist-sol
│           Dimensions:  ()
│           Data variables:
│               tas      float64 8B -0.008726
├── Group: /CanESM5
│   │   Dimensions:  (year: 31)
│   │   Coordinates:
│   │     * year     (year) int64 248B 1990 1991 1992 1993 1994 ... 2017 2018 2019 2020
│   │   Attributes:
│   │       institution:  CCCma
│   ├── Group: /CanESM5/historical
│   │       Dimensions:  ()
│   │       Data variables:
│   │           tas      float64 8B 0.8786
│   ├── Group: /CanESM5/hist-GHG
│   │       Dimensions:  ()
│   │       Data variables:
│   │           tas      float64 8B 1.172
│   ├── Group: /CanESM5/hist-aer
│   │       Dimensions:  ()
│   │       Data variables:
│   │           tas      float64 8B -0.4195
│   └── Group: /CanESM5/hist-sol
│           Dimensions:  ()
│           Data variables:
│               tas      float64 8B 0.0005992
└── Group: /MIROC6
    │   Dimensions:  (year: 31)
    │   Coordinates:
    │     * year     (year) int64 248B 1990 1991 1992 1993 1994 ... 2017 2018 2019 2020
    │   Attributes:
    │       institution:  MIROC
    ├── Group: /MIROC6/historical
    │       Dimensions:  ()
    │       Data variables:
    │           tas      float64 8B 0.8967
    ├── Group: /MIROC6/hist-GHG
    │       Dimensions:  ()
    │       Data variables:
    │           tas      float64 8B 1.227
    ├── Group: /MIROC6/hist-aer
    │       Dimensions:  ()
    │       Data variables:
    │           tas      float64 8B -0.4269
    └── Group: /MIROC6/hist-nat

**Gotcha with inherited coords.** Here `year` lives on the root. Reducing over it tree-wide works on the leaves,
but the empty model nodes keep a stray (subsetted) `year` coordinate (visible on the model nodes in `recent` above), which then clashes with the full
`year` on the root when you combine trees:

In [35]:
naive = dt.sel(year=slice(1850, 1900)).mean("year")
print("stray year on /MIROC6:", "year" in naive["MIROC6"].to_dataset(inherit=False).coords)

try:
    dt - naive
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])

stray year on /MIROC6: True
ValueError: group '/ACCESS-ESM1-5' is not aligned with its parents:


Fixes: do the reduction with a guarded `map_over_datasets` (as `baseline` does in section 6), so empty nodes pass
through untouched, or don't put shared dimension coords on the root in the first place.

In [36]:
# isel / chunk / load / resample / etc. all work the same way
dt.isel(member=0)["MIROC6/historical"].ds

<xarray.DatasetView> Size: 3kB
Dimensions:  (year: 171)
Coordinates:
  * year     (year) int64 1kB 1850 1851 1852 1853 1854 ... 2017 2018 2019 2020
    member   int64 8B 1
Data variables:
    tas      (year) float64 1kB 0.001865 -0.06814 -0.05378 ... 1.225 1.281

### `__getitem__` takes paths (and still takes variable names on a node)

In [37]:
print(type(dt["CanESM5/hist-aer"]).__name__)        # DataTree node
print(type(dt["CanESM5/hist-aer/tas"]).__name__)    # DataArray
print(type(dt["CanESM5/hist-aer"]["tas"]).__name__) # DataArray

DataTree
DataArray
DataArray


### `to_dict()` / `from_dict()` use path keys

In [38]:
dd = dt.to_dict()
list(dd)[:6]

['/',
 '/ACCESS-ESM1-5',
 '/CanESM5',
 '/MIROC6',
 '/ACCESS-ESM1-5/historical',
 '/ACCESS-ESM1-5/hist-GHG']

In [39]:
xr.DataTree.from_dict(dd).identical(dt)

True

### `to_zarr` / `to_netcdf` write each node as a group; `xr.open_datatree` reads them back

In [40]:
import tempfile, os, warnings
warnings.filterwarnings("ignore", message="Consolidated metadata")   # zarr v3 noise
path = os.path.join(tempfile.mkdtemp(), "toy_lesfmip.zarr")
dt.to_zarr(path)

reopened = xr.open_datatree(path, engine="zarr")
print(reopened.groups[:5])
print("round-trip identical:", reopened.load().identical(dt))

('/', '/ACCESS-ESM1-5', '/CanESM5', '/MIROC6', '/ACCESS-ESM1-5/hist-aer')
round-trip identical: True


You can also open a single group: `xr.open_dataset(path, engine='zarr', group='CanESM5/hist-GHG')`.

### `equals` / `identical` / `copy` operate on the whole tree

In [41]:
dt2 = dt.copy(deep=True)
dt2["MIROC6"].attrs["institution"] = "changed"

print("equals   :", dt.equals(dt2))      # data + structure
print("identical:", dt.identical(dt2))   # also attrs and names

equals   : True
identical: False


## 8. Recipes

### `xr.group_subtrees`: loop over several trees at matching paths
Like `map_over_datasets`, but you write the loop yourself (useful when you're not building a tree back).

In [42]:
for path, (node, base) in xr.group_subtrees(dt, baseline):
    if "tas" in node.data_vars:
        change = float((node.ds.tas - base.ds.tas).sel(year=2020).mean())
        print(f"{path:26s} 2020 anomaly = {change:+.2f} K")

ACCESS-ESM1-5/historical   2020 anomaly = +1.12 K
ACCESS-ESM1-5/hist-GHG     2020 anomaly = +1.58 K
ACCESS-ESM1-5/hist-aer     2020 anomaly = -0.40 K
ACCESS-ESM1-5/hist-sol     2020 anomaly = +0.03 K
CanESM5/historical         2020 anomaly = +1.15 K
CanESM5/hist-GHG           2020 anomaly = +1.55 K
CanESM5/hist-aer           2020 anomaly = -0.34 K
CanESM5/hist-sol           2020 anomaly = +0.21 K
MIROC6/historical          2020 anomaly = +1.11 K
MIROC6/hist-GHG            2020 anomaly = +1.29 K
MIROC6/hist-aer            2020 anomaly = -0.47 K


### Flatten a tree into a single Dataset (from earlier)

In [43]:
def tree_to_dataset(dt: xr.DataTree, dims: list[str], **concat_kwargs) -> xr.Dataset:
    '''Collapse a DataTree into a Dataset, mapping each path level to a dimension.'''
    leaves = [node for node in dt.leaves if node.data_vars]   # data_vars, not has_data (which counts coords)
    keys = [tuple(node.relative_to(dt).split("/")) for node in leaves]

    if any(len(k) != len(dims) for k in keys):
        raise ValueError(f"Not all leaves are at depth {len(dims)}")

    concat_kwargs.setdefault("join", "outer")
    ds = xr.concat([node.to_dataset() for node in leaves], dim="_stacked", **concat_kwargs)

    idx = pd.MultiIndex.from_tuples(keys, names=dims)
    ds = ds.assign_coords(xr.Coordinates.from_pandas_multiindex(idx, "_stacked"))
    return ds.unstack("_stacked")


flat = tree_to_dataset(dt, dims=["model", "experiment"])
flat.tas

<xarray.DataArray 'tas' (member: 3, year: 171, model: 3, experiment: 4)> Size: 49kB
-0.02002 -0.1173 -0.08564 0.01886 -0.2017 ... 1.292 1.381 -0.5315 nan 1.187
Coordinates:
  * member      (member) int64 24B 1 2 3
  * year        (year) int64 1kB 1850 1851 1852 1853 ... 2017 2018 2019 2020
  * model       (model) object 24B 'ACCESS-ESM1-5' 'CanESM5' 'MIROC6'
  * experiment  (experiment) object 32B 'hist-GHG' 'hist-aer' ... 'historical'

In [44]:
# ragged combos come back as NaN
flat.tas.sel(model="MIROC6", experiment="hist-sol").isnull().all().item()

True

### Typical pipeline
select with `match`/`filter` → reduce with tree-wide methods → flatten at the end

In [45]:
result = tree_to_dataset(
    (dt.match("*/hist-*") - baseline.match("*/hist-*"))
      .sel(year=slice(1995, 2014))
      .mean(["year", "member"]),
    dims=["model", "experiment"],
)
result.tas.to_pandas().round(2)

experiment,hist-GHG,hist-aer,hist-sol
model,,,
ACCESS-ESM1-5,1.17,-0.37,-0.01
CanESM5,1.14,-0.39,-0.03
MIROC6,1.18,-0.39,NaN


## 9. Deprecated names (avoid)

These still exist but emit `FutureWarning`s:

| deprecated | use instead |
|---|---|
| `.lineage` | `.parents` (plus self if you need it: `(node, *node.parents)`) |
| `.iter_lineage()` | `.parents` |
| `.ancestors` | `tuple(reversed(node.parents))` (root first) |